# 新算法对比 Benchmark: NOTEARS / Greedy-CD / FGES

本 Notebook 在与 `manual_dag_validate_all_algorithms.ipynb` 相同的 9 组人工 DAG 上，
测试以下新增算法，并与已有的 `cd_A`、`cd_B`、`cd_BOmega`、GES、SP 进行对比：

- **NOTEARS** (Zheng et al. 2018) — 连续松弛 + 增广拉格朗日，via `gcastle`
- **Greedy-CD** (`dag_greedy_A_epoch`) — 本项目贪婪坐标下降，via `coordinate_descent/cd_greedy_A.py`
- **FGES** (Ramsey et al.) — Fast Greedy Equivalence Search，via `pytetrad`（需 JDK 21+）

评估指标（每个 experiment × n_repeats 次独立采样）：
- `mec_match_mean`：是否落在真实 MEC 内的比例
- `exact_match_mean`：精确图结构匹配比例
- `cpdag_shd_mean / std`：CPDAG-SHD（越小越好）
- `runtime_sec_mean`：平均运行时间（秒）

In [6]:
import os
import sys
import time
import json
import platform
import logging
from datetime import datetime
from itertools import permutations

import numpy as np
import pandas as pd

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("Repo root:", repo_root)

# --- 项目内模块 ---
from MEC import is_in_markov_equiv_class, get_skeleton, find_v_structures
from SCM_data import generate_scm_from_BN
# epoch variants
from coordinate_descent.coordinate0 import dag_coordinate_descent_l0_epoch as cd_A
from coordinate_descent.cd_B import dag_coordinate_descent_B_epoch as cd_B
from coordinate_descent.cd_B_Omega import dag_coordinate_descent_BOmega_epoch as cd_BOmega
# non-epoch variants
from coordinate_descent.coordinate0 import dag_coordinate_descent_l0 as cd_A_noepoch
from coordinate_descent.cd_B import dag_coordinate_descent_B as cd_B_noepoch
from coordinate_descent.cd_B_Omega import dag_coordinate_descent_BOmega as cd_BOmega_noepoch

# --- Greedy-CD (epoch + non-epoch) ---
greedy_cd_src = os.path.join(repo_root, "coordinate_descent")
if greedy_cd_src not in sys.path:
    sys.path.append(greedy_cd_src)
try:
    from cd_greedy_A import dag_greedy_A_epoch as greedy_cd_fit
    from cd_greedy_A import dag_greedy_A as greedy_cd_noepoch_fit
    HAS_GREEDY_CD = True
    print("Greedy-CD : OK")
except Exception as e:
    HAS_GREEDY_CD = False
    print("Greedy-CD unavailable:", e)

# --- NOTEARS (gcastle) ---
NOTEARS_IMPORT_ERROR = None
try:
    logging.disable(logging.INFO)
    from castle.algorithms import Notears as _Notears
    logging.disable(logging.NOTSET)
    HAS_NOTEARS = True
    print("NOTEARS   : OK")
except Exception as e:
    logging.disable(logging.NOTSET)
    HAS_NOTEARS = False
    NOTEARS_IMPORT_ERROR = e
    print("NOTEARS unavailable, will skip:", e)

# --- FGES (pytetrad, requires JDK 21+) ---
FGES_IMPORT_ERROR = None
try:
    import pytetrad.tools.TetradSearch as _ts
    _probe_df = pd.DataFrame(np.eye(2), columns=["x0", "x1"])
    _s = _ts.TetradSearch(_probe_df)
    _s.set_verbose(False); _s.use_sem_bic(); _s.run_fges()
    del _probe_df, _s
    HAS_FGES = True
    print("FGES      : OK")
except Exception as e:
    HAS_FGES = False
    FGES_IMPORT_ERROR = e
    print("FGES unavailable, will skip:", e)

# --- GOLEM ---
GOLEM_IMPORT_ERROR = None
try:
    golem_src = os.path.join(repo_root, "golemMain", "src")
    if golem_src not in sys.path:
        sys.path.append(golem_src)
    from golem import golem as golem_fit
    HAS_GOLEM = True
    print("GOLEM     : OK")
except Exception as e:
    HAS_GOLEM = False
    GOLEM_IMPORT_ERROR = e
    print("GOLEM unavailable, will skip:", e)

# --- GES (causallearn) ---
GES_IMPORT_ERROR = None
try:
    from causallearn.search.ScoreBased.GES import ges as ges_fit
    HAS_GES = True
    print("GES       : OK")
except Exception as e:
    HAS_GES = False
    GES_IMPORT_ERROR = e
    print("GES unavailable, will skip:", e)

# --- CPDAG-SHD backend ---
try:
    if os.path.join(repo_root, "toolbox") not in sys.path:
        sys.path.append(os.path.join(repo_root, "toolbox"))
    from cdt.metrics import SHD_CPDAG as _SHD_CPDAG
    HAS_CPDAG_SHD = True
except Exception as e:
    _SHD_CPDAG = None
    HAS_CPDAG_SHD = False
    print("CPDAG-SHD backend unavailable, using fallback:", e)

# ============================================================
# Configuration
# ============================================================
CFG = {
    "n_samples"            : 10000,
    "n_repeats"            : 50,
    "master_seed"          : 42,
    "threshold"            : 0.05,
    "k"                    : None,
    "dag_tol"              : 1e-8,
    # epoch CD
    "epochs_a"             : 500,
    "epochs_b"             : 500,
    "epochs_bomega"        : 500,
    "epochs_greedy"        : 500,
    "lambda_l0_list"       : [0.0, 0.1, 0.2],
    "tol"                  : 1e-4,
    "patience"             : 50,
    "min_epochs"           : 100,
    "eps_omega"            : 1e-8,
    # non-epoch CD (fixed passes)
    "T_noepoch"            : 100000,
    # NOTEARS
    "notears_lambda1"      : 0.1,
    "notears_loss_type"    : "l2",
    "notears_threshold"    : 0.3,
    # FGES
    "fges_penalty_discount": 1.0,
    # GOLEM
    "run_golem"            : True,
    "golem_num_iter"       : 100000,
    "golem_learning_rate"  : 1e-3,
    "golem_lambda1_ev"     : 2e-2,
    "golem_lambda1_nv"     : 2e-3,
    "golem_lambda2"        : 5.0,
    # flags
    "run_sp"               : True,
    "run_ges"              : True,
    "run_notears"          : True,
    "run_greedy_cd"        : True,
    "run_fges"             : True,
    # output
    "out_dir"              : os.path.join(repo_root, "experiments", "results"),
    "tag"                  : "test_new_algorithms_20260329",
}
os.makedirs(CFG["out_dir"], exist_ok=True)

ALGORITHM_ORDER = [
    "cd_A_noepoch_l0_0.0",      "cd_A_noepoch_l0_0.1",      "cd_A_noepoch_l0_0.2",
    "greedy_cd_noepoch_l0_0.0", "greedy_cd_noepoch_l0_0.1", "greedy_cd_noepoch_l0_0.2",
    "notears",
    "fges",
    "golem_ev"
]

print("\nConfig ready. n_repeats =", CFG["n_repeats"])
print("Algorithms (%d total):" % len(ALGORITHM_ORDER))
for a in ALGORITHM_ORDER:
    print("  ", a)

Python : 3.12.3
Platform: Windows-11-10.0.26200-SP0
numpy  : 1.26.4
pandas : 2.2.2
Repo root: c:\Users\super\DAG
Greedy-CD : OK
NOTEARS unavailable, will skip: No module named 'castle'
FGES unavailable, will skip: No module named 'pytetrad'
GOLEM     : OK
GES       : OK

Config ready. n_repeats = 50
Algorithms (9 total):
   cd_A_noepoch_l0_0.0
   cd_A_noepoch_l0_0.1
   cd_A_noepoch_l0_0.2
   greedy_cd_noepoch_l0_0.0
   greedy_cd_noepoch_l0_0.1
   greedy_cd_noepoch_l0_0.2
   notears
   fges
   golem_ev


In [7]:
# ============================================================
# 9 组人工 DAG 实验（与 manual_dag_validate_all_algorithms 完全一致）
# ============================================================

def get_manual_experiments():
    exps = []
    exps.append({'experiment_id': 1, 'name': 'd=3, A→B←C',
                 'B_true': np.array([[0,1,0],[0,0,0],[0,2,0]], dtype=float),
                 'Omega_true': np.diag([1,2,3]).astype(float)})
    exps.append({'experiment_id': 2, 'name': 'd=3, A→B→C',
                 'B_true': np.array([[0,1,0],[0,0,3],[0,0,0]], dtype=float),
                 'Omega_true': np.diag([1,3,4]).astype(float)})
    exps.append({'experiment_id': 3, 'name': 'd=3, A→B→C + A→C',
                 'B_true': np.array([[0,1,2],[0,0,3],[0,0,0]], dtype=float),
                 'Omega_true': np.diag([5,4,3]).astype(float)})
    exps.append({'experiment_id': 4, 'name': 'd=4, A→B, B→C, B→D',
                 'B_true': np.array([[0,2,0,0],[0,0,3,4],[0,0,0,0],[0,0,0,0]], dtype=float),
                 'Omega_true': np.diag([1,4,3,2]).astype(float)})
    exps.append({'experiment_id': 5, 'name': 'd=4, A→C, A→D, B→C, B→D',
                 'B_true': np.array([[0,0,2,3],[0,0,3,4],[0,0,0,0],[0,0,0,0]], dtype=float),
                 'Omega_true': np.diag([2,4,3,5]).astype(float)})
    exps.append({'experiment_id': 6, 'name': 'd=4, A→D, B→D, C→D',
                 'B_true': np.array([[0,0,0,1],[0,0,0,3],[0,0,0,5],[0,0,0,0]], dtype=float),
                 'Omega_true': np.diag([5,4,3,2]).astype(float)})
    exps.append({'experiment_id': 7, 'name': 'd=5, e=4, |v|=0',
                 'B_true': np.array([[0,1,0,2,0],[0,0,3,0,4],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=float),
                 'Omega_true': np.diag([1,2,3,2,1]).astype(float)})
    exps.append({'experiment_id': 8, 'name': 'd=5, e=4, |v|=1',
                 'B_true': np.array([[0,0,1,2,0],[0,0,0,2,3],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=float),
                 'Omega_true': np.diag([1,2,3,2,1]).astype(float)})
    exps.append({'experiment_id': 9, 'name': 'd=5, e=4, |v|=2',
                 'B_true': np.array([[0,0,0,1,0],[0,0,0,2,3],[0,0,0,0,4],[0,0,0,0,0],[0,0,0,0,0]], dtype=float),
                 'Omega_true': np.diag([1,2,3,2,1]).astype(float)})
    return exps

print('Loaded', len(get_manual_experiments()), 'experiments.')
for e in get_manual_experiments():
    print(f"  [{e['experiment_id']}] {e['name']}")

Loaded 9 experiments.
  [1] d=3, A→B←C
  [2] d=3, A→B→C
  [3] d=3, A→B→C + A→C
  [4] d=4, A→B, B→C, B→D
  [5] d=4, A→C, A→D, B→C, B→D
  [6] d=4, A→D, B→D, C→D
  [7] d=5, e=4, |v|=0
  [8] d=5, e=4, |v|=1
  [9] d=5, e=4, |v|=2


In [8]:
# ============================================================
# 辅助函数
# ============================================================

def weight_to_binary_adj(W: np.ndarray, threshold: float) -> np.ndarray:
    G = (np.abs(W) > threshold).astype(int)
    np.fill_diagonal(G, 0)
    return G


def cpdag_shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    if HAS_CPDAG_SHD:
        try:
            return float(_SHD_CPDAG(G_true.astype(int), G_est.astype(int)))
        except Exception:
            pass
    skel_true = get_skeleton(G_true)
    skel_est  = get_skeleton(G_est)
    skeleton_diff = int(np.sum(np.abs(skel_true - skel_est)) // 2)
    v_true = find_v_structures(G_true)
    v_est  = find_v_structures(G_est)
    v_diff = len(v_true.symmetric_difference(v_est))
    return float(skeleton_diff + v_diff)


def evaluate_algorithm(G_true: np.ndarray, G_est: np.ndarray) -> dict:
    return {
        'mec_match' : int(is_in_markov_equiv_class(G_true, G_est)),
        'exact_match': int(np.array_equal(G_true, G_est)),
        'cpdag_shd' : cpdag_shd_score(G_true, G_est),
    }


def sp_estimate_W(X: np.ndarray) -> np.ndarray:
    """SP (exhaustive permutation search); only feasible for d <= 8."""
    from numpy.linalg import inv, cholesky, LinAlgError
    n, p = X.shape
    if p > 8:
        raise ValueError(f'SP exhaustive search is too expensive for d={p}; skipping.')
    Sigma_hat = np.cov(X, rowvar=False)

    def l0_norm(U, threshold=0.05):
        return int(np.sum(np.abs(U) > threshold))

    best_score, best_W = np.inf, None
    for perm in permutations(range(p)):
        P = np.eye(p)[list(perm)]
        Sigma_perm = P @ Sigma_hat @ P.T
        try:
            Theta = inv(Sigma_perm)
            L = cholesky(Theta)
            diag_L = np.diag(L)
            sqrt_Omega = np.diag(1.0 / diag_L)
            W = np.eye(p) - L @ sqrt_Omega
            score = l0_norm(W)
            if score < best_score:
                best_score = score
                best_W = P.T @ W @ P
        except LinAlgError:
            continue
    if best_W is None:
        raise RuntimeError('SP failed to find a valid structure.')
    np.fill_diagonal(best_W, 0.0)
    return best_W


def ges_graph_to_adj(g: np.ndarray) -> np.ndarray:
    """Convert causallearn GES graph matrix (−1/1 encoding) to 0/1 adjacency."""
    g = np.asarray(g)
    d = g.shape[0]
    A = np.zeros((d, d), dtype=int)
    for i in range(d):
        for j in range(i + 1, d):
            a, b = g[i, j], g[j, i]
            if   a == -1 and b ==  1: A[i, j] = 1
            elif a ==  1 and b == -1: A[j, i] = 1
            elif a == -1 and b == -1: A[i, j] = A[j, i] = 1
            elif a != 0 or  b != 0:
                A[i, j] = int(a != 0)
                A[j, i] = int(b != 0)
    np.fill_diagonal(A, 0)
    return A


def fges_matrix_to_adj(df_result: 'pd.DataFrame') -> np.ndarray:
    """
    Convert pytetrad get_graph_to_matrix() result to 0/1 adjacency.

    Encoding (default): null=0, circle=1, arrow=2, tail=3
    For a directed edge i→j:  mat[i,j]=2 (arrow at j), mat[j,i]=3 (tail at i)
    For an undirected edge i—j: mat[i,j]=3, mat[j,i]=3 (both tails)
    """
    mat = df_result.values
    d   = mat.shape[0]
    G   = np.zeros((d, d), dtype=int)
    ARROW, TAIL = 2, 3
    for i in range(d):
        for j in range(i + 1, d):
            a, b = int(mat[i, j]), int(mat[j, i])
            if   a == ARROW and b == TAIL:  G[i, j] = 1          # i → j
            elif a == TAIL  and b == ARROW: G[j, i] = 1          # j → i
            elif a == TAIL  and b == TAIL:  G[i, j] = G[j, i] = 1  # undirected
            elif a != 0 or b != 0:          G[i, j] = G[j, i] = 1  # other edges
    np.fill_diagonal(G, 0)
    return G


def _append_row(rows, exp_id, exp_name, repeat_id, seed, alg,
                runtime_sec, G_true, G_est):
    m = evaluate_algorithm(G_true, G_est)
    rows.append({
        'experiment_id': int(exp_id),
        'experiment'   : exp_name,
        'repeat_id'    : int(repeat_id),
        'seed'         : int(seed),
        'algorithm'    : alg,
        'mec_match'    : m['mec_match'],
        'exact_match'  : m['exact_match'],
        'cpdag_shd'    : m['cpdag_shd'],
        'runtime_sec'  : float(runtime_sec),
    })


print('Helper functions defined.')

Helper functions defined.


In [9]:
# ============================================================
# 主 benchmark 函数
# ============================================================

def run_new_alg_benchmark(cfg: dict, n_repeats: int = 50, save_tag: str = None):
    if save_tag is None:
        save_tag = cfg['tag']

    exps = get_manual_experiments()
    rng  = np.random.default_rng(cfg['master_seed'])
    rows, skip_logs = [], []

    for exp in exps:
        exp_id   = exp['experiment_id']
        exp_name = exp['name']
        B_true   = np.asarray(exp['B_true'],   dtype=float)
        Omega_true = np.asarray(exp['Omega_true'], dtype=float)
        N        = np.diag(Omega_true).copy()
        d        = B_true.shape[0]
        seeds    = rng.integers(0, 10**9, size=n_repeats)
        G_true   = weight_to_binary_adj(B_true, threshold=0.0)

        for r_idx, seed in enumerate(seeds, start=1):
            X, _, _, _ = generate_scm_from_BN(
                B_true.T, n_samples=cfg['n_samples'], N=N, seed=int(seed)
            )
            S = X.T @ X / X.shape[0]

            # ---- CD-A variants ----
            for lam in cfg['lambda_l0_list']:
                tag = f'{lam:.1f}'

                t0 = time.perf_counter()
                _, G_A, _, _ = cd_A(
                    S=S, n_epochs=cfg['epochs_a'], seed=int(seed),
                    threshold=cfg['threshold'], lambda_l0=lam,
                    tol=cfg['tol'], patience=cfg['patience'],
                    min_epochs=cfg['min_epochs'], verbose=False,
                )
                _append_row(rows, exp_id, exp_name, r_idx, seed,
                            f'cd_A_l0_{tag}', time.perf_counter()-t0, G_true, G_A)

                t0 = time.perf_counter()
                _, G_B, _, _, _ = cd_B(
                    S=S, n_epochs=cfg['epochs_b'], seed=int(seed),
                    threshold=cfg['threshold'], lambda_l0=lam,
                    k=cfg['k'], dag_tol=cfg['dag_tol'],
                    tol=cfg['tol'], patience=cfg['patience'],
                    min_epochs=cfg['min_epochs'], verbose=False,
                )
                _append_row(rows, exp_id, exp_name, r_idx, seed,
                            f'cd_B_l0_{tag}', time.perf_counter()-t0, G_true, G_B)

                t0 = time.perf_counter()
                _, G_BOm, _, _, _ = cd_BOmega(
                    S=S, Omega=np.eye(d), n_epochs=cfg['epochs_bomega'],
                    seed=int(seed), threshold=cfg['threshold'], lambda_l0=lam,
                    k=cfg['k'], dag_tol=cfg['dag_tol'],
                    tol=cfg['tol'], patience=cfg['patience'],
                    min_epochs=cfg['min_epochs'], eps_omega=cfg['eps_omega'],
                    verbose=False,
                )
                _append_row(rows, exp_id, exp_name, r_idx, seed,
                            f'cd_BOmega_l0_{tag}', time.perf_counter()-t0, G_true, G_BOm)

            # ---- CD-A non-epoch ----
            for lam in cfg['lambda_l0_list']:
                tag = f'{lam:.1f}'
                t0 = time.perf_counter()
                A_ne, G_A_ne, _ = cd_A_noepoch(
                    S=S, T=cfg['T_noepoch'], seed=int(seed),
                    threshold=cfg['threshold'], lambda_l0=lam,
                )
                _append_row(rows, exp_id, exp_name, r_idx, seed,
                            f'cd_A_noepoch_l0_{tag}', time.perf_counter()-t0, G_true, G_A_ne)

            # ---- CD-B non-epoch ----
            for lam in cfg['lambda_l0_list']:
                tag = f'{lam:.1f}'
                t0 = time.perf_counter()
                _, G_B_ne, _, _ = cd_B_noepoch(
                    S=S, T=cfg['T_noepoch'], seed=int(seed),
                    threshold=cfg['threshold'], lambda_l0=lam,
                    k=cfg['k'], dag_tol=cfg['dag_tol'],
                )
                _append_row(rows, exp_id, exp_name, r_idx, seed,
                            f'cd_B_noepoch_l0_{tag}', time.perf_counter()-t0, G_true, G_B_ne)

            # ---- CD-BOmega non-epoch ----
            for lam in cfg['lambda_l0_list']:
                tag = f'{lam:.1f}'
                t0 = time.perf_counter()
                _, G_BOm_ne, _, _ = cd_BOmega_noepoch(
                    S=S, Omega=np.eye(d), T=cfg['T_noepoch'], seed=int(seed),
                    threshold=cfg['threshold'], lambda_l0=lam,
                    k=cfg['k'], dag_tol=cfg['dag_tol'], eps_omega=cfg['eps_omega'],
                )
                _append_row(rows, exp_id, exp_name, r_idx, seed,
                            f'cd_BOmega_noepoch_l0_{tag}', time.perf_counter()-t0, G_true, G_BOm_ne)

                        # ---- Greedy-CD ----
            if cfg.get('run_greedy_cd', True) and HAS_GREEDY_CD:
                for lam in cfg['lambda_l0_list']:
                    tag = f'{lam:.1f}'
                    try:
                        t0 = time.perf_counter()
                        _, G_gcd, _, _ = greedy_cd_fit(
                            S=S, n_epochs=cfg['epochs_greedy'],
                            seed=int(seed), threshold=cfg['threshold'],
                            lambda_l0=lam, tol=cfg['tol'],
                            patience=cfg['patience'],
                            min_epochs=cfg['min_epochs'], verbose=False,
                        )
                        _append_row(rows, exp_id, exp_name, r_idx, seed,
                                    f'greedy_cd_l0_{tag}', time.perf_counter()-t0,
                                    G_true, G_gcd)
                    except Exception as e:
                        skip_logs.append({
                            'experiment_id': exp_id, 'repeat_id': r_idx,
                            'seed': int(seed), 'algorithm': f'greedy_cd_l0_{tag}',
                            'reason': str(e),
                        })

            # ---- Greedy-CD non-epoch ----
            if cfg.get('run_greedy_cd', True) and HAS_GREEDY_CD:
                for lam in cfg['lambda_l0_list']:
                    tag = f'{lam:.1f}'
                    try:
                        t0 = time.perf_counter()
                        A_gne, G_gne, _ = greedy_cd_noepoch_fit(
                            S=S, T=cfg['T_noepoch'], seed=int(seed),
                            threshold=cfg['threshold'], lambda_l0=lam,
                        )
                        _append_row(rows, exp_id, exp_name, r_idx, seed,
                                    f'greedy_cd_noepoch_l0_{tag}', time.perf_counter()-t0,
                                    G_true, G_gne)
                    except Exception as e:
                        skip_logs.append({
                            'experiment_id': exp_id, 'repeat_id': r_idx,
                            'seed': int(seed), 'algorithm': f'greedy_cd_noepoch_l0_{tag}',
                            'reason': str(e),
                        })

                        # ---- NOTEARS ----
            if cfg.get('run_notears', True) and HAS_NOTEARS:
                try:
                    logging.disable(logging.INFO)
                    t0 = time.perf_counter()
                    model = _Notears(
                        lambda1    = cfg['notears_lambda1'],
                        loss_type  = cfg['notears_loss_type'],
                        w_threshold= cfg['notears_threshold'],
                    )
                    model.learn(X)
                    elapsed = time.perf_counter() - t0
                    logging.disable(logging.NOTSET)
                    G_nt = model.causal_matrix.astype(int)
                    np.fill_diagonal(G_nt, 0)
                    _append_row(rows, exp_id, exp_name, r_idx, seed,
                                'notears', elapsed, G_true, G_nt)
                except Exception as e:
                    logging.disable(logging.NOTSET)
                    skip_logs.append({
                        'experiment_id': exp_id, 'repeat_id': r_idx,
                        'seed': int(seed), 'algorithm': 'notears', 'reason': str(e),
                    })

            # ---- FGES ----
            if cfg.get('run_fges', True) and HAS_FGES:
                try:
                    cols = [f'x{i}' for i in range(d)]
                    df_X = pd.DataFrame(X, columns=cols).astype('float64')
                    t0 = time.perf_counter()
                    search = _ts.TetradSearch(df_X)
                    search.set_verbose(False)
                    search.use_sem_bic(
                        penalty_discount=cfg.get('fges_penalty_discount', 1.0)
                    )
                    search.run_fges()
                    elapsed = time.perf_counter() - t0
                    G_fges = fges_matrix_to_adj(search.get_graph_to_matrix())
                    _append_row(rows, exp_id, exp_name, r_idx, seed,
                                'fges', elapsed, G_true, G_fges)
                except Exception as e:
                    skip_logs.append({
                        'experiment_id': exp_id, 'repeat_id': r_idx,
                        'seed': int(seed), 'algorithm': 'fges', 'reason': str(e),
                    })

            # ---- GOLEM ----
            if cfg.get('run_golem', True) and HAS_GOLEM:
                try:
                    t0 = time.perf_counter()
                    B_ev = golem_fit(
                        X, lambda_1=cfg['golem_lambda1_ev'],
                        lambda_2=cfg['golem_lambda2'], equal_variances=True,
                        num_iter=cfg['golem_num_iter'],
                        learning_rate=cfg['golem_learning_rate'], seed=int(seed),
                    )
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'golem_ev',
                                time.perf_counter()-t0, G_true,
                                weight_to_binary_adj(B_ev, cfg['threshold']))

                    t0 = time.perf_counter()
                    B_nv = golem_fit(
                        X, lambda_1=cfg['golem_lambda1_nv'],
                        lambda_2=cfg['golem_lambda2'], equal_variances=False,
                        num_iter=cfg['golem_num_iter'],
                        learning_rate=cfg['golem_learning_rate'], seed=int(seed),
                    )
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'golem_nv',
                                time.perf_counter()-t0, G_true,
                                weight_to_binary_adj(B_nv, cfg['threshold']))
                except Exception as e:
                    skip_logs.append({
                        'experiment_id': exp_id, 'repeat_id': r_idx,
                        'seed': int(seed), 'algorithm': 'golem_ev/nv', 'reason': str(e),
                    })

            # ---- SP ----
            if cfg.get('run_sp', True):
                try:
                    t0 = time.perf_counter()
                    B_sp = sp_estimate_W(X)
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'sp',
                                time.perf_counter()-t0, G_true,
                                weight_to_binary_adj(B_sp, cfg['threshold']))
                except Exception as e:
                    skip_logs.append({
                        'experiment_id': exp_id, 'repeat_id': r_idx,
                        'seed': int(seed), 'algorithm': 'sp', 'reason': str(e),
                    })

            # ---- GES ----
            if cfg.get('run_ges', True) and HAS_GES:
                try:
                    t0 = time.perf_counter()
                    rec = ges_fit(X)
                    G_ges = ges_graph_to_adj(rec['G'].graph)
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'ges',
                                time.perf_counter()-t0, G_true, G_ges)
                except Exception as e:
                    skip_logs.append({
                        'experiment_id': exp_id, 'repeat_id': r_idx,
                        'seed': int(seed), 'algorithm': 'ges', 'reason': str(e),
                    })

        print(f'[Experiment {exp_id}/9] {exp_name}  done  (rows so far: {len(rows)})')

    # ---- 汇总 ----
    df_trials = pd.DataFrame(rows)
    if len(df_trials) == 0:
        print('WARNING: no results collected!')
        return df_trials, pd.DataFrame(), skip_logs

    df_summary = (
        df_trials
        .groupby(['experiment_id', 'experiment', 'algorithm'], as_index=False)
        .agg(
            mec_match_mean  =('mec_match',   'mean'),
            exact_match_mean=('exact_match', 'mean'),
            cpdag_shd_mean  =('cpdag_shd',   'mean'),
            cpdag_shd_std   =('cpdag_shd',   'std'),
            runtime_sec_mean=('runtime_sec', 'mean'),
            runs            =('algorithm',   'size'),
        )
    )

    full_index = pd.MultiIndex.from_product(
        [sorted(df_trials['experiment_id'].unique()), ALGORITHM_ORDER],
        names=['experiment_id', 'algorithm'],
    )
    exp_name_map = (
        df_summary[['experiment_id', 'experiment']]
        .drop_duplicates()
        .set_index('experiment_id')['experiment']
        .to_dict()
    )
    df_summary = (
        df_summary
        .set_index(['experiment_id', 'algorithm'])
        .reindex(full_index)
        .reset_index()
    )
    df_summary['experiment'] = df_summary['experiment_id'].map(exp_name_map)
    df_summary = df_summary[[
        'experiment_id', 'experiment', 'algorithm',
        'mec_match_mean', 'exact_match_mean',
        'cpdag_shd_mean', 'cpdag_shd_std',
        'runtime_sec_mean', 'runs',
    ]]

    # ---- 存档 ----
    ts_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    trials_path  = os.path.join(cfg['out_dir'], f'{save_tag}_trials_{ts_str}.csv')
    summary_path = os.path.join(cfg['out_dir'], f'{save_tag}_summary_{ts_str}.csv')
    skips_path   = os.path.join(cfg['out_dir'], f'{save_tag}_skips_{ts_str}.json')
    df_trials.to_csv(trials_path,  index=False)
    df_summary.to_csv(summary_path, index=False)
    # latest symlinks (overwrites)
    df_trials.to_csv( os.path.join(cfg['out_dir'], f'{save_tag}_trials.csv'),  index=False)
    df_summary.to_csv(os.path.join(cfg['out_dir'], f'{save_tag}_summary.csv'), index=False)
    with open(skips_path, 'w', encoding='utf-8') as f:
        json.dump(skip_logs, f, ensure_ascii=False, indent=2)

    print()
    print(f'HAS_GREEDY_CD : {HAS_GREEDY_CD}')
    print(f'HAS_NOTEARS   : {HAS_NOTEARS}')
    print(f'HAS_FGES      : {HAS_FGES}')
    print(f'HAS_GOLEM     : {HAS_GOLEM}')
    print(f'HAS_GES       : {HAS_GES}')
    print(f'Trials  saved : {trials_path}')
    print(f'Summary saved : {summary_path}')
    print(f'Skips   saved : {skips_path}')

    return df_trials, df_summary, skip_logs


print('Benchmark function defined.')

Benchmark function defined.


In [10]:
# ============================================================
# Smoke run（每个 experiment 仅 1 次）—— 快速验证流程
# GOLEM 关闭以加快速度
# ============================================================

CFG_SMOKE = dict(CFG)
CFG_SMOKE['n_repeats'] = 1
CFG_SMOKE['run_golem'] = False
CFG_SMOKE['tag']       = CFG['tag'] + '_smoke'

df_trials_smoke, df_summary_smoke, skip_logs_smoke = run_new_alg_benchmark(
    cfg=CFG_SMOKE,
    n_repeats=CFG_SMOKE['n_repeats'],
    save_tag=CFG_SMOKE['tag'],
)

print('\nSmoke trials shape:', df_trials_smoke.shape)
print('Smoke skip count  :', len(skip_logs_smoke))
display(df_summary_smoke)

[Experiment 1/9] d=3, A→B←C  done  (rows so far: 26)
[Experiment 2/9] d=3, A→B→C  done  (rows so far: 52)
[Experiment 3/9] d=3, A→B→C + A→C  done  (rows so far: 78)
[Experiment 4/9] d=4, A→B, B→C, B→D  done  (rows so far: 104)
[Experiment 5/9] d=4, A→C, A→D, B→C, B→D  done  (rows so far: 130)
[Experiment 6/9] d=4, A→D, B→D, C→D  done  (rows so far: 156)
[Experiment 7/9] d=5, e=4, |v|=0  done  (rows so far: 182)
[Experiment 8/9] d=5, e=4, |v|=1  done  (rows so far: 208)
[Experiment 9/9] d=5, e=4, |v|=2  done  (rows so far: 234)

HAS_GREEDY_CD : True
HAS_NOTEARS   : False
HAS_FGES      : False
HAS_GOLEM     : True
HAS_GES       : True
Trials  saved : c:\Users\super\DAG\experiments\results\test_new_algorithms_20260329_smoke_trials_20260330_004710.csv
Summary saved : c:\Users\super\DAG\experiments\results\test_new_algorithms_20260329_smoke_summary_20260330_004710.csv
Skips   saved : c:\Users\super\DAG\experiments\results\test_new_algorithms_20260329_smoke_skips_20260330_004710.json

Smoke 

,experiment_id,experiment,algorithm,mec_match_mean,exact_match_mean,cpdag_shd_mean,cpdag_shd_std,runtime_sec_mean,runs
0,1,"d=3, A→B←C",cd_A_noepoch_l0_0.0,1.0,1.0,0.0,NaN,40.469618,1.0
1,1,"d=3, A→B←C",cd_A_noepoch_l0_0.1,1.0,1.0,0.0,NaN,41.134761,1.0
2,1,"d=3, A→B←C",cd_A_noepoch_l0_0.2,1.0,1.0,0.0,NaN,41.293194,1.0
3,1,"d=3, A→B←C",greedy_cd_noepoch_l0_0.0,0.0,0.0,2.0,NaN,10.795802,1.0
4,1,"d=3, A→B←C",greedy_cd_noepoch_l0_0.1,0.0,0.0,2.0,NaN,11.514547,1.0
...,...,...,...,...,...,...,...,...,...
76,9,"d=5, e=4, |v|=2",greedy_cd_noepoch_l0_0.1,0.0,0.0,4.0,NaN,11.795223,1.0
77,9,"d=5, e=4, |v|=2",greedy_cd_noepoch_l0_0.2,0.0,0.0,4.0,NaN,12.001767,1.0
78,9,"d=5, e=4, |v|=2",notears,NaN,NaN,NaN,NaN,NaN,NaN
79,9,"d=5, e=4, |v|=2",fges,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# ============================================================
# 正式实验（n_repeats=50）
# ============================================================

df_trials, df_summary, skip_logs = run_new_alg_benchmark(
    cfg=CFG,
    n_repeats=CFG['n_repeats'],
    save_tag=CFG['tag'],
)

print('\nFull trials shape:', df_trials.shape)
print('Skip count        :', len(skip_logs))
display(df_summary)



R Python Error Output 
-----------------------

C:\Users\super\AppData\Local\Temp\cdt_CPDAG_7159abff-fc4d-47cc-a0d5-579662db7441\result.csv not found.
[Experiment 1/9] d=3, A→B←C  done  (rows so far: 1400)


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 跨 9 个 experiment 的算法总体均值对比
# ============================================================

df_overall = (
    df_trials
    .groupby('algorithm', as_index=False)
    .agg(
        mec_match_mean  =('mec_match',   'mean'),
        exact_match_mean=('exact_match', 'mean'),
        cpdag_shd_mean  =('cpdag_shd',   'mean'),
        runtime_sec_mean=('runtime_sec', 'mean'),
        runs            =('algorithm',   'size'),
    )
)

# 按 ALGORITHM_ORDER 排序
alg_order_map = {alg: i for i, alg in enumerate(ALGORITHM_ORDER)}
df_overall['_order'] = df_overall['algorithm'].map(alg_order_map).fillna(999)
df_overall = df_overall.sort_values('_order').drop(columns='_order').reset_index(drop=True)

print('Overall algorithm comparison (averaged across all 9 experiments):')
display(df_overall.style.format({
    'mec_match_mean'  : '{:.3f}',
    'exact_match_mean': '{:.3f}',
    'cpdag_shd_mean'  : '{:.3f}',
    'runtime_sec_mean': '{:.4f}',
}).highlight_max(
    subset=['mec_match_mean', 'exact_match_mean'], color='lightgreen'
).highlight_min(
    subset=['cpdag_shd_mean', 'runtime_sec_mean'], color='lightyellow'
))

In [ ]:
# ============================================================
# 跳过记录分析
# ============================================================

if skip_logs:
    df_skips = pd.DataFrame(skip_logs)
    print(f'Total skips: {len(df_skips)}')
    display(df_skips.groupby('algorithm')['reason'].value_counts().rename('count').reset_index())
else:
    print('No skips — all algorithms ran successfully on all repeats.')